In [1]:
import pandas as pd 
import pickle
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report,roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib

In [2]:
data = pd.read_csv("cleaned_data.csv")

In [3]:
data.head()

,VendorNumber,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Avg_PurchasePrice,Num_Products,Num_Brands,PaymentDelay,Risk
0,105,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,35.710000,1,1,43,1
1,4466,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,9.370000,3,2,45,1
2,388,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,21.320000,1,1,38,0
3,480,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,14.336467,367,81,24,0
4,516,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,8.041461,89,29,36,0


In [4]:
data.drop("PaymentDelay",axis=1,inplace=True)

In [5]:
data.head()

,VendorNumber,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Avg_PurchasePrice,Num_Products,Num_Brands,Risk
0,105,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,35.710000,1,1,1
1,4466,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,9.370000,3,2,1
2,388,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,21.320000,1,1,0
3,480,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,14.336467,367,81,0
4,516,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,8.041461,89,29,0


In [6]:
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])
data["PODate"] = pd.to_datetime(data["PODate"])
data["PayDate"] = pd.to_datetime(data["PayDate"])
data["InvoiceMonth"]=data["InvoiceDate"].dt.month
data["POMonth"]=data["PODate"].dt.month
data["PayMonth"] = data["PayDate"].dt.month
data.drop("InvoiceDate",axis=1,inplace=True)
data.drop("PODate",axis=1,inplace=True)
data.drop("PayDate",axis=1,inplace=True)
data.drop("VendorNumber",axis=1,inplace=True)

In [7]:
train_data,test_data = train_test_split(data,test_size=0.2,random_state=42)

In [8]:
train_data_label = train_data["Risk"]
train_data.drop("Risk",axis=1,inplace=True)
test_data_label = test_data["Risk"]
test_data.drop("Risk",axis=1,inplace=True)

In [9]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4434 entries, 2609 to 860
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   PONumber           4434 non-null   int64  
 1   Quantity           4434 non-null   int64  
 2   Dollars            4434 non-null   float64
 3   Freight            4434 non-null   float64
 4   Avg_PurchasePrice  4434 non-null   float64
 5   Num_Products       4434 non-null   int64  
 6   Num_Brands         4434 non-null   int64  
 7   InvoiceMonth       4434 non-null   int32  
 8   POMonth            4434 non-null   int32  
 9   PayMonth           4434 non-null   int32  
dtypes: float64(3), int32(3), int64(4)
memory usage: 329.1 KB


In [10]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1109 entries, 4564 to 70
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   PONumber           1109 non-null   int64  
 1   Quantity           1109 non-null   int64  
 2   Dollars            1109 non-null   float64
 3   Freight            1109 non-null   float64
 4   Avg_PurchasePrice  1109 non-null   float64
 5   Num_Products       1109 non-null   int64  
 6   Num_Brands         1109 non-null   int64  
 7   InvoiceMonth       1109 non-null   int32  
 8   POMonth            1109 non-null   int32  
 9   PayMonth           1109 non-null   int32  
dtypes: float64(3), int32(3), int64(4)
memory usage: 82.3 KB


In [11]:
num_columns = train_data.select_dtypes(include=["int64","float64"]).columns.tolist()
cat_columns = train_data.select_dtypes(include="int32").columns.tolist()
print(num_columns)
print(cat_columns)

['PONumber', 'Quantity', 'Dollars', 'Freight', 'Avg_PurchasePrice', 'Num_Products', 'Num_Brands']
['InvoiceMonth', 'POMonth', 'PayMonth']


In [12]:
#Pipelines 

num_pipeline = Pipeline([
    ("scaler",StandardScaler())
])

cat_pipeline = Pipeline([
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

general_pipeline = ColumnTransformer([
    ("num_pipe",num_pipeline,num_columns),
    ("cat_pipe",cat_pipeline,cat_columns)
])

In [13]:
train_processed_data = general_pipeline.fit_transform(train_data)
test_processed_data = general_pipeline.transform(test_data)

In [14]:
#Logistic classifier 

log_model = LogisticRegression(class_weight="balanced")
log_model.fit(train_processed_data,train_data_label)
pred_log_model = log_model.predict(test_processed_data)

#Random Forest classifier 

rand_model = RandomForestClassifier(class_weight="balanced",n_estimators=200,random_state=42)
rand_model.fit(train_processed_data,train_data_label)
pred_rand_model = rand_model.predict(test_processed_data)

#Decision Tree Classifier 

tree_model = DecisionTreeClassifier(class_weight="balanced")
tree_model.fit(train_processed_data,train_data_label)
pred_tree_model = tree_model.predict(test_processed_data)

#XGB 

xgb_model = XGBClassifier(n_estimators=200,random_state=42)
xgb_model.fit(train_processed_data,train_data_label)
pred_xgb_model = xgb_model.predict(test_processed_data)

In [15]:
#Accuracy Score 

print("Accuracy Score of Log Model is ",accuracy_score(pred_log_model,test_data_label))
print("Accuracy Score of Rand Model is ",accuracy_score(pred_rand_model,test_data_label))
print("Accuracy Score of Tree Model is ",accuracy_score(pred_tree_model,test_data_label))
print("Accuracy Score of Xgb Model is ",accuracy_score(pred_xgb_model,test_data_label))

Accuracy Score of Log Model is  0.5518485121731289
Accuracy Score of Rand Model is  0.7718665464382326
Accuracy Score of Tree Model is  0.6862037871956718
Accuracy Score of Xgb Model is  0.7709648331830478


In [16]:
#classification report 

print("Classification Report of Log Model is ",classification_report(pred_log_model,test_data_label))
print("Classification Report of Rand Model is ",classification_report(pred_rand_model,test_data_label))
print("Classification Report of Tree Model is ",classification_report(pred_tree_model,test_data_label))
print("Classification Report of Xgb Model is ",classification_report(pred_xgb_model,test_data_label))

Classification Report of Log Model is                precision    recall  f1-score   support

           0       0.58      0.79      0.67       630
           1       0.46      0.24      0.31       479

    accuracy                           0.55      1109
   macro avg       0.52      0.51      0.49      1109
weighted avg       0.53      0.55      0.51      1109

Classification Report of Rand Model is                precision    recall  f1-score   support

           0       0.96      0.79      0.87      1046
           1       0.11      0.43      0.18        63

    accuracy                           0.77      1109
   macro avg       0.53      0.61      0.52      1109
weighted avg       0.91      0.77      0.83      1109

Classification Report of Tree Model is                precision    recall  f1-score   support

           0       0.79      0.80      0.80       855
           1       0.31      0.30      0.30       254

    accuracy                           0.69      1109
   macro 

In [17]:
#Rouc Score

print("Roc score of Log Model is ",roc_auc_score(pred_log_model,test_data_label))
print("Roc score of Rand Model is ",roc_auc_score(pred_rand_model,test_data_label))
print("Roc score of Tree Model is ",roc_auc_score(pred_tree_model,test_data_label))
print("Roc score of Xgb Model is ",roc_auc_score(pred_xgb_model,test_data_label))

Roc score of Log Model is  0.5139858170129569
Roc score of Rand Model is  0.6105572248019668
Roc score of Tree Model is  0.5488073859188654
Roc score of Xgb Model is  0.6386063418173025


In [23]:
joblib.dump(general_pipeline,"risk_model_preprocessing.pkl")

['risk_model_preprocessing.pkl']

In [24]:
joblib.dump(xgb_model,"risk_model.pkl")

['risk_model.pkl']

In [19]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4434 entries, 2609 to 860
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   PONumber           4434 non-null   int64  
 1   Quantity           4434 non-null   int64  
 2   Dollars            4434 non-null   float64
 3   Freight            4434 non-null   float64
 4   Avg_PurchasePrice  4434 non-null   float64
 5   Num_Products       4434 non-null   int64  
 6   Num_Brands         4434 non-null   int64  
 7   InvoiceMonth       4434 non-null   int32  
 8   POMonth            4434 non-null   int32  
 9   PayMonth           4434 non-null   int32  
dtypes: float64(3), int32(3), int64(4)
memory usage: 329.1 KB


In [20]:
test_data.head(20)

,PONumber,Quantity,Dollars,Freight,Avg_PurchasePrice,Num_Products,Num_Brands,InvoiceMonth,POMonth,PayMonth
4564,12694,48,352.95,1.73,15.184815,24,3,11,10,12
1616,9749,34773,225706.96,1196.25,6.943917,2730,248,4,4,5
4861,12955,70,634.11,2.85,17.451690,103,6,11,11,12
230,8383,104,987.34,4.64,9.441111,9,4,1,1,2
2042,10205,4314,31768.74,152.49,9.181274,469,45,5,5,6
4522,12619,1965,22699.66,118.04,9.890000,31,3,11,10,12
4291,12340,14554,89941.85,458.70,13.184683,187,34,10,9,11
5316,13409,2647,21465.16,113.77,10.187766,90,35,12,12,2
4139,12258,35,462.49,2.54,14.530000,4,4,10,9,11
381,8497,83,2028.52,9.53,24.916667,12,3,2,1,3


In [21]:
test_data_label.head(20)

4564    0
1616    0
4861    0
230     0
2042    0
4522    0
4291    1
5316    1
4139    1
381     0
4250    0
4238    0
4832    0
2728    1
5137    0
2195    0
439     0
5222    0
3265    0
373     0
Name: Risk, dtype: int64